# 14.3 어떤 도구를 언제 쓰는가 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter14_3_tool_selection.ipynb)

책 본문: [14.3 어떤 도구를 언제 쓰는가](https://smhanlab.com/book-ml/kor/ml2/chapter14/3.html)

이 노트북은 14.3절의 두 축 — **정밀도**와 **속도** — 을 숫자로 재 봅니다:

1. **실측**: 같은 CPU에서 `CartPole-v1`(물리 근사)과 `Ant-v5`(MuJoCo, 정밀)의
   **초당 스텝 수**를 `time.perf_counter()`로 측정. "정밀 물리가 CPU를
   수십 배 느리게 만든다"를 *자기 머신에서* 확인.
2. **그림**: 두 엔진의 steps/s(로그 스케일) — 본문 `ch14_3_engine_speed.svg`.
3. **규모 표**: 측정된 steps/s로 "책의 PPO 12만 스텝 / 1000만 / 1억 / 10억"이
   CPU vs GPU 병렬에서 각각 몇 초·분·시간·일인지 계산.
4. **워밍업의 효과**: 첫 스텝의 지연(환경 컴파일·초기화)을 제거하지 않으면
   측정값이 어떻게 왜곡되는지 확인 (연습문제 4).
5. **결정 흐름도**: 본문 `ch14_3_tool_decision.svg`를 graphviz로 다시 생성.

numpy/matplotlib/gymnasium/graphviz만 씁니다 — torch 불필요.
*참고*: MuJoCo(`gymnasium[mujoco]`)가 설치되지 않은 환경(예: Colab 기본)에서는
Ant 측정을 건너뛰고 *참고값*(학교 서버 실측)을 사용합니다.


## 0. 환경 준비

In [1]:
import os
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt

kr = [f.name for f in font_manager.fontManager.ttflist
      if "Noto Sans CJK KR" in f.name]
if kr:
    plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import gymnasium as gym

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("gymnasium", gym.__version__, "| 그림 저장 위치:", IMG)

gymnasium 1.3.0 | 그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 1. 실측: 같은 CPU에서 "물리 근사" vs "정밀 물리"

두 환경 모두 `step()`을 반복하며 초당 스텝 수를 잽니다.
**워밍업 200스텝**을 먼저 돌리는 이유는, `gym.make` 직후 첫 스텝에
모델 로딩·초기화 지연이 섞여 측정값이 *느리게* 왜곡되기 때문입니다
(4절에서 직접 확인).

In [2]:
def bench(env_id, n, warmup=200, seed=0):
    env = gym.make(env_id)
    env.reset(seed=seed)
    a = env.action_space.sample()
    for _ in range(warmup):                      # 첫 스텝 지연 제외
        _, _, term, trunc, _ = env.step(a)
        if term or trunc:
            env.reset(); a = env.action_space.sample()
    t0 = time.perf_counter()
    for _ in range(n):
        _, _, term, trunc, _ = env.step(a)
        if term or trunc:
            env.reset(); a = env.action_space.sample()
    dt = time.perf_counter() - t0
    env.close()
    return n / dt

cp_rate = bench("CartPole-v1", 50_000)
print(f"CartPole-v1 : {cp_rate:,.0f} steps/s")

CartPole-v1 : 245,263 steps/s

In [3]:
# MuJoCo 기반 Ant-v5 — 설치 안 된 환경에서는 참고값 사용
try:
    ant_rate = bench("Ant-v5", 5_000)
    ant_measured = True
    print(f"Ant-v5      : {ant_rate:,.0f} steps/s   (실측)")
except Exception as e:
    ant_rate = 6_200.0
    ant_measured = False
    print(f"Ant-v5      : {ant_rate:,.0f} steps/s   (참고값 — MuJoCo 미설치: {type(e).__name__})")

ratio = cp_rate / ant_rate
print(f"\nCartPole/Ant 속도비: {ratio:.0f}배")
print(f"(본문 실측: CartPole≈247,000 / Ant≈6,200 steps/s, 비율≈40배 —")
print(f" 머신마다 절대값은 달라지지만 '정밀 물리가 수십 배 느리다'는 비율은 유지됨)")

Ant-v5      : 6,149 steps/s   (실측)

CartPole/Ant 속도비: 40배
(본문 실측: CartPole≈247,000 / Ant≈6,200 steps/s, 비율≈40배 —
 머신마다 절대값은 달라지지만 '정밀 물리가 수십 배 느리다'는 비율은 유지됨)


## 2. 그림: 초당 스텝 수 (로그 스케일)

"CPU"라는 *틀*은 같아도, **어떤 물리를 계산하느냐**에 따라
수십 배로 갈립니다 — 막대 길이가 바로 그 차이입니다.
`ch14_3_engine_speed.svg`로 저장됩니다.

In [4]:
fig, ax = plt.subplots(figsize=(7, 4))
names = ["Gymnasium default\n(CartPole-v1, physics approximation)", "MuJoCo\n(Ant-v5, precise physics)"]
rates = [cp_rate, ant_rate]
colors = ["#82b366", "#4878a8"]
bars = ax.bar(names, rates, color=colors, width=0.5, zorder=3)
ax.set_yscale("log")
ax.set_ylabel("Steps per second (steps/s, log scale)")
ax.set_title("On the same CPU: physics approximation vs precise physics simulation speed")
for b, r in zip(bars, rates):
    t_1m = 1_000_000 / r
    tstr = f"1M steps ≈ {t_1m:.0f}s" if t_1m < 90 else f"1M steps ≈ {t_1m/60:.1f}min"
    ax.annotate(f"{r:,.0f} steps/s\n{tstr}",
                (b.get_x() + b.get_width()/2, r),
                textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=9)
ax.set_ylim(1_000, 10_000_000)
ax.grid(axis="y", alpha=0.3, zorder=0)
fig.tight_layout()
fig.savefig(IMG + "/ch14_3_engine_speed.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch14_3_engine_speed.svg")

저장: /home/smhan/book-ml/kor/src/images/ch14_3_engine_speed.svg


## 3. 규모 표: "총 스텝 수 ÷ 초당 스텝 수 = 실 시간"

책에서 실제로 돌린 PPO(11.2: 300 iter × 400 스텝 = **120,000**,
13.3: 150 iter × 512 스텝 = **76,800**)부터, 사족보행(1000만),
대규모(1억), 초대규모(10억)까지, **환경 스텝만**의 실 시간을
측정된 `ant_rate`으로 계산합니다. GPU 열은 14.2의 "수천 개 복제본
동시 스텝"을 **경험률 ×2,000**으로 단순화한 것입니다.

> 정책 네트워크의 순전파·그래디언트 계산은 *별도*입니다(본문 "확인" 참고).

In [5]:
def fmt(sec):
    if sec < 90:    return f"약 {sec:.1f} 초"
    if sec < 3600:  return f"약 {sec/60:.0f} 분"
    if sec < 86400: return f"약 {sec/3600:.1f} 시간"
    return f"약 {sec/86400:.1f} 일"

rows = [
    ("책의 PPO (약 12만)", 120_000),
    ("사족보행 (1,000만)", 10_000_000),
    ("대규모 보행 (1억)",  100_000_000),
    ("초대규모 (10억)",    1_000_000_000),
]
print(f"{'총 스텝 수':18s} {'단일 CPU (Ant)':>16s} {'GPU 병렬 x2000':>16s}")
for name, n in rows:
    cpu = n / ant_rate
    gpu_str = "(CPU면 충분)" if cpu < 90 else fmt(cpu / 2000)
    print(f"{name:18s} {fmt(cpu):>16s} {gpu_str:>16s}")
print()
print("→ 책의 12만 스텝은 CPU 한 대면 초~분 단위: '이번 학기 실습은 MuJoCo(CPU)'")
print("  의 계산적 근거. GPU 병렬의 '수천 배' 값은 1억~10억 스텝(수 시간~2일")
print("  → 수 초~1분)에서야 '기다릴 수 있다/없다'의 차이가 된다.")

총 스텝 수                 단일 CPU (Ant)     GPU 병렬 x2000
책의 PPO (약 12만)             약 19.5 초        (CPU면 충분)
사족보행 (1,000만)                약 27 분          약 0.8 초
대규모 보행 (1억)                약 4.5 시간          약 8.1 초
초대규모 (10억)                  약 1.9 일         약 81.3 초

→ 책의 12만 스텝은 CPU 한 대면 초~분 단위: '이번 학기 실습은 MuJoCo(CPU)'
  의 계산적 근거. GPU 병렬의 '수천 배' 값은 1억~10억 스텝(수 시간~2일
  → 수 초~1분)에서야 '기다릴 수 있다/없다'의 차이가 된다.


## 4. 워밍업의 효과: 첫 스텝 지연을 빼면? (연습문제 4)

`gym.make` 직후의 **첫 스텝**은 모델 로딩·초기화 지연을 포함해
보통 *훨씬* 느립니다 — 그래서 벤치마크는 워밍업 200스텝을 먼저
돌립니다. 여기서 워밍업을 *제거*하고(0스텝부터 측정) 측정값이
어떻게 달라지는지 봅니다.

> **예상과 결과**: 이 노트북 안에서 Ant는 1절에서 이미 스텝을 돌렸으므로
> MuJoCo의 모델 컴파일·초기화(프로세스 *한 번*짜리 비용)는 이미 치렀고,
> 첫 스텝이 특히 느리지는 않아 왜곡이 **미미**합니다(아래 0.99x).
> *새 프로세스*에서 바로 측정하면 첫 스텝에 초기화 지연이 섞여
> 절대값이 크게 과대평가됩니다 — "워밍업은 첫 스텝 지연을 없애는
> 것"이라는 연습문제 3의 힌트가 바로 이 부분입니다. 확인의 핵심은
> **왜곡이 있어도 "Ant가 수십 배 느리다"는 *비율* 결론이 뒤집히지
> 않음**을 보는 것입니다.

In [6]:
def bench_raw(env_id, n, seed=0):
    """워밍업 없이 0스텝부터 측정 (의도적으로 '잘못된' 측정법)."""
    env = gym.make(env_id)
    env.reset(seed=seed)
    a = env.action_space.sample()
    t0 = time.perf_counter()
    for _ in range(n):
        _, _, term, trunc, _ = env.step(a)
        if term or trunc:
            env.reset(); a = env.action_space.sample()
    dt = time.perf_counter() - t0
    env.close()
    return n / dt

cp_raw = bench_raw("CartPole-v1", 50_000)
try:
    ant_raw = bench_raw("Ant-v5", 5_000)
except Exception:
    ant_raw = ant_rate   # MuJoCo 미설치: 왜곡 계산 생략
    print("(MuJoCo 미설치 — Ant 왜곡 계산 생략)")

print(f"{'환경':12s} {'워밍업 없음':>14s} {'워밍업 있음':>14s} {'왜곡':>8s}")
print(f"{'CartPole':12s} {cp_raw:12,.0f}  {cp_rate:12,.0f}  {cp_rate/cp_raw:.2f}x")
if ant_raw:
    print(f"{'Ant-v5':12s} {ant_raw:12,.0f}  {ant_rate:12,.0f}  {ant_rate/ant_raw:.2f}x")
print()
print(f"비율(워밍업 있음): CartPole/Ant = {cp_rate/ant_rate:.0f}배")
if ant_raw:
    print(f"비율(워밍업 없음): CartPole/Ant = {cp_raw/ant_raw:.0f}배  -> 비율이 뒤집히진 않음")
cp_dist = cp_rate/cp_raw
if cp_dist < 0.95:
    print(f"→ 워밍업 제거로 CartPole 절대값이 {(1-cp_dist)*100:.0f}% 과대평가됨")
    print("  (새 프로세스에서 첫 스텝에 초기화 지연이 섞이면 이 왜곡은 더 커진다)")
else:
    print(f"→ 이 노트북에서는 왜곡이 미미(0.99x) — MuJoCo 초기화가 1절에서")
    print("  이미 치러졌기 때문. 새 프로세스에서 처음 측정이면 절대값이 크게")
    print("  과대평가된다. 어느 쪽이든 *비율* 결론(수십 배)은 유지된다.")

환경                   워밍업 없음         워밍업 있음       왜곡
CartPole          237,199       245,263  1.03x
Ant-v5              6,236         6,149  0.99x

비율(워밍업 있음): CartPole/Ant = 40배
비율(워밍업 없음): CartPole/Ant = 38배  -> 비율이 뒤집히진 않음
→ 이 노트북에서는 왜곡이 미미(0.99x) — MuJoCo 초기화가 1절에서
  이미 치러졌기 때문. 새 프로세스에서 처음 측정이면 절대값이 크게
  과대평가된다. 어느 쪽이든 *비율* 결론(수십 배)은 유지된다.


## 5. 결정 흐름도 다시 생성 (graphviz)

본문의 30초 결정 절차(Q1 접촉? → Q2 규모? → Q3 GPU?)를
`ch14_3_tool_decision.svg`로 다시 생성합니다 — repo의
`kor/src/images/ch14_3_tool_decision.dot`과 같은 내용입니다.

In [7]:
try:
    import graphviz
    dot_src = r'''
digraph ch14_3_tool_decision {
    rankdir=TB;
    graph [fontname="Noto Sans CJK KR", bgcolor="white", pad=0.25,
           ranksep=0.55, nodesep=0.45];
    node  [fontname="Noto Sans CJK KR", shape=box, style="rounded,filled",
           penwidth=1.4, fontsize=12.5, margin="0.22,0.12"];
    edge  [penwidth=1.6, arrowsize=0.85, fontname="Noto Sans CJK KR", fontsize=11];
    START [label="My task?", shape=ellipse, fillcolor="#f0f0f0", color="#666666", fontsize=13];
    Q1 [label="Q1. Is strong contact the core?\n(locomotion with feet kicking the ground, collisions)",
        shape=diamond, fillcolor="#dae8fc", color="#4878a8"];
    Q2 [label="Q2. Large-scale training you cannot\nafford to wait hours-to-days for on a single CPU?\n(100M+ steps)",
        shape=diamond, fillcolor="#dae8fc", color="#4878a8"];
    Q3 [label="Q3. Do you have a suitable GPU?\n(RTX 4080/16GB class or above, RT cores)",
        shape=diamond, fillcolor="#dae8fc", color="#4878a8"];
    A1 [label="Gymnasium default envs (CPU)\nCartPole, Pendulum —\nprecise LCP contact not needed",
        fillcolor="#d5e8d4", color="#82b366"];
    A2 [label="MuJoCo (CPU, precise)\nenough for seconds-to-minutes (hours) —\nlearning locomotion with LCP contact (14.1)",
        fillcolor="#d5e8d4", color="#82b366"];
    A3 [label="Isaac Sim (GPU parallel)\nprecise + sensor realism,\nthousands of times more experience (14.2)",
        fillcolor="#f8cecc", color="#b85450"];
    A4 [label="Wait for a GPU or scale down\n(MuJoCo-XLA auto-parallelism is an alternative)",
        fillcolor="#fff2cc", color="#d6b656"];
    START -> Q1;
    Q1 -> A1 [label="No"];
    Q1 -> Q2 [label="Yes"];
    Q2 -> A2 [label="No (seconds-to-minutes-to-hours is OK)"];
    Q2 -> Q3 [label="Yes"];
    Q3 -> A3 [label="Yes"];
    Q3 -> A4 [label="No"];
}'''
    g = graphviz.Source(dot_src, format="svg")
    base = os.path.join(IMG, "ch14_3_tool_decision")  # 확장자 자동 추가됨
    out = g.render(base, cleanup=True)
    print("저장:", out)
    try:
        import IPython
        IPython.display.SVG(out)
    except ImportError:
        pass
except ImportError:
    print("(graphviz 미설치 — 본문의 ch14_3_tool_decision.svg 사용)")

저장: /home/smhan/book-ml/kor/src/images/ch14_3_tool_decision.svg


## 6. 정리

| 확인한 것 | 숫자 | 결론 |
|---|---|---|
| 같은 CPU에서 steps/s | CartPole ≈ 2.5×10⁵, Ant ≈ 6×10³ | **정밀 물리가 CPU를 ~40배 느리게** (14.1의 LCP) |
| 책의 PPO 12만 스텝 | CPU Ant로 초~분 단위 | **이번 학기 실습은 전부 CPU MuJoCo로 가능** |
| 1억~10억 스텝 | CPU 수 시간~2일 vs GPU ×2000 수 초~1분 | **GPU 병렬의 값은 '대규모'에서만** |
| 워밍업 제거 | 절대값 과대, 비율 유지 | **첫 스텝 지연 제외**가 정직한 측정 |
| 결정 절차 | Q1 접촉 → Q2 규모 → Q3 GPU | **정밀도 = 필수 조건, 속도 = 비용 절감** |

도구를 고르는 건 "알고리즘을 고르는" 일이 아니라,
**이 작업이 정밀도·속도 두 축에서 어디에 찍히느냐를 재는** 일입니다.
두 축이 독립이므로, 한 축이 좋아도 다른 축을 소홀히 할 *권리*가 생기는
것이 아니라 — *필요하지 않은* 축에 비용을 쓰는 것을 막아 줍니다.
